In [138]:
"""Airstest 初始化"""

import logging
import sys
import pyperclip
from pathlib import Path
from subprocess import run, PIPE
parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

from airtest.core.api import *

logging.getLogger("airtest").setLevel(logging.CRITICAL)

path = str(Path().resolve().parent.parent)
if path not in sys.path:
    sys.path.append(path)

from utility.base.adc_api import *

connect_cmd_ = """
ssh -L {local_port}:localhost:5555 root@{ip} -p {port} \
-i {id_rsa} -Nf -o ServerAliveInterval=60 -o StrictHostKeyChecking=no -o CheckHostIP=no \
-o UserKnownHostsFile=/dev/null
adb connect localhost:{local_port}
"""
id_rsa_1 = "/Users/vip/aic-jymadc-keypair-test/id_rsa_1"
id_rsa_2 = "/Users/vip/aic-jymadc-keypair-test/id_rsa_2"

# 连接设备
# ip_port, prefix, id_rsa = "111.62.192.12:5690", 1, id_rsa_1  # 本地1
ip_port, prefix, id_rsa = "111.62.192.12:5647", 1, id_rsa_1  # 本地2
# ip_port, prefix, id_rsa = "111.6.24.11:5999", 1, id_rsa_1  # 本地3
# ip_port, prefix, id_rsa = "111.62.192.12:5686", 1, id_rsa_1  # 宋斌的
# ip_port, prefix, id_rsa = "111.62.192.12:5719", 2, id_rsa_1  # 抖音达人
# ip_port, prefix, id_rsa = "111.62.192.12_5912", 2, id_rsa_1

ip = port = ""
if ":" in ip_port:
    ip, port = ip_port.split(":")
elif "_" in ip_port:
    ip, port = ip_port.split("_")

local_port = f"{prefix}{port[-4:]}"
connect_cmd = connect_cmd_.format(ip=ip, port=port, local_port=local_port, id_rsa=id_rsa)
run(connect_cmd, **parameter)

# 初始化
local_device = f"localhost:{local_port}"
auto_setup(devices=[
    f"android://localhost:5037/{local_device}",
    # f"android:///46287227",
    # "iOS:///http://220.176.196.253:15078"  # 电信 iOS 设备1 串流 15054
    # "iOS:///http://220.176.196.253:15118"  # 电信 iOS 设备2 串流 15094
    # "iOS:///http://113.195.134.221:15078"  # 联通 iOS 设备1 串流 15054
    # "iOS:///http://113.195.134.221:15118"  # 联通 iOS 设备2 串流 15094
])
DEV = device()
print(f"{DEV.uuid}")
pyperclip.copy(f"{DEV.uuid}")

localhost:15647


In [140]:
"""Airstest 高清截屏"""

import datetime
import pyperclip
from pathlib import Path
from subprocess import run, Popen, call
import cv2

# gameDir = "8788_yingjiao"         # 明日方舟
# gameDir = "13765_shiyue"          # 长安幻想
# gameDir = "15574_xingjia"         # 最强祖师
# gameDir = "6586_common"           # 第五人格
# gameDir = "2717_common"           # 率土之滨
# gameDir = "9207_jiuyou"           # 三国志·战略版jiuyou
# gameDir = "9207_lingxi"           # 三国志·战略版lingxi
gameDir = "5799_steam"            # 绝地求生PUBG-PC
# gameDir = "ios_23016_shengchang"  # 向僵尸开炮iOS
# gameDir = "9999018_qufu"          # 抖音达人
# gameDir = "2416_common"           # 王者荣耀

if gameDir.endswith("_qufu"):
    save_path = Path().resolve().parent.parent / f"{gameDir}/{gameDir}_qufu.air"
elif gameDir.startswith("ios"):
    save_path = Path().resolve().parent.parent / f"iOS/{gameDir}/{gameDir}_index"
else:
    save_path = Path().resolve().parent.parent / f"{gameDir}/{gameDir}_index.air"

# 保存到下载
# save_path = Path.home().resolve() / "Downloads"

def screenshot_hd():
    for i in range(1):
        filename = f"{datetime.datetime.now().strftime('tpl%Y%m%d%H%M%S')}.png"
        filepath = f"{save_path}/{filename}"
        image = DEV.snapshot()
        cv2.imwrite(filepath, image)
        pyperclip.copy(f"\"{filename}\"")
        print(f"Save to {filepath}")
        Popen(['open', filepath])

screenshot_hd()

Save to /Users/vip/adc-sc/5799_steam/5799_steam_index.air/tpl20260414175723.png


In [142]:
"""Scrcpy 投屏"""

from subprocess import Popen, run, PIPE
from pathlib import Path
from multiprocessing import Process
# from support_func import get_window_titles_applescript
import re

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

id_rsa_1 = "/Users/vip/aic-jymadc-keypair-test/id_rsa_1"
id_rsa_2 = "/Users/vip/aic-jymadc-keypair-test/id_rsa_2"

# titles = get_window_titles_applescript()
# sub_titles = list(map(lambda x: x.split("-", 1)[1].strip(), titles))

devices = [
    # 线下设备
    # ("111.62.192.12:5690", id_rsa_1, 1),  # 本地1
    # ("111.62.192.12:5647", id_rsa_1, 1),  # 本地2
    # ("111.6.24.11:5999", id_rsa_1, 1),  # 本地3
    # ("111.62.192.12:5686", id_rsa_1, 1),  # 宋斌的
    # ("120.232.102.68:19678", id_rsa_1, 1),  # 无尽冬日

    # 线上设备
    # ("111.62.202.85:5965", id_rsa_1, 2), # 预发区服更新
    # ("111.6.24.11:5913", id_rsa_1, 2),   # 线上区服更新1
    # ("111.6.24.11:5912", id_rsa_1, 2),   # 线上区服更新2
    # ("111.6.24.11:5911", id_rsa_1, 2),   # 线上区服更新3
    # ("111.6.24.11:5990", id_rsa_1, 2),   # 预发三战: 小米
    # ("111.6.24.11:5991", id_rsa_1, 2),   # 预发三战: 九游1
    # ("111.6.24.11:5993", id_rsa_1, 2),   # 预发三战: 九游2
    # ("111.6.24.11:5994", id_rsa_1, 2),   # 预发三战: 九游3
    # ("111.6.24.11:5992", id_rsa_1, 2),   # 预发三战: 犀游、灵犀
    # ("111.62.202.85:5645", id_rsa_1, 2),  # 小米
    # ("111.6.24.11:5958", id_rsa_1, 2),  # 话题采集
    # ("111.62.192.12:5719", id_rsa_1, 2),  # 抖音达人
    # ("111.6.24.11:5996", id_rsa_1, 2),   # 预发
    # ("111.62.192.12:5910", id_rsa_1, 2),

    # 云游设备
    # ("120.232.194.43:10078", id_rsa_2, 2),
    ("106.117.208.69_13638", id_rsa_2, 2),
]
# res = run("adb devices", **parameter)
# devices_list_re = re.compile(r"localhost:(\d+)")
# devices_list = res.stdout.strip()
# port_list = devices_list_re.findall(devices_list)

connect_cmd_ = """
ssh -L {local_port}:localhost:5555 root@{ip} -p {port} \
-i {private_key} -Nf -o ServerAliveInterval=60 -o StrictHostKeyChecking=no -o CheckHostIP=no \
-o UserKnownHostsFile=/dev/null
adb connect localhost:{local_port}
scrcpy -s localhost:{local_port} -b 12m --always-on-top -m 1500  --no-audio --window-title "{title}"
""".strip()

for ip_port, private_key, prefix in devices:
    ip = port = ""
    if ":" in ip_port:
        ip, port = ip_port.split(":")
    elif "_" in ip_port:
        ip, port = ip_port.split("_")

    local_port = f"{prefix}{port[-4:]}"
    title = f"Scrcpy {ip_port} -> localhost:{local_port}"
    # connect_cmd_ += " --no-control" # 只读模式

    if "--no-control" in connect_cmd_:
        title += " (只读)"
    connect_cmd = connect_cmd_.format(ip=ip, port=port, local_port=local_port, private_key=private_key, title=title)
    print(connect_cmd)
    print(title)
    # filename = Path.home() / f"Desktop/{ip}_{port}投屏.sh"
    # filename.write_text(connect_cmd)
    # filename.chmod(0o700)
    # Popen(['/System/Applications/Utilities/Terminal.app/Contents/MacOS/Terminal', filename])
    Process(target=run, args=(connect_cmd,), kwargs=parameter).start()

ssh -L 23638:localhost:5555 root@106.117.208.69 -p 13638 -i /Users/vip/aic-jymadc-keypair-test/id_rsa_2 -Nf -o ServerAliveInterval=60 -o StrictHostKeyChecking=no -o CheckHostIP=no -o UserKnownHostsFile=/dev/null
adb connect localhost:23638
scrcpy -s localhost:23638 -b 12m --always-on-top -m 1500  --no-audio --window-title "Scrcpy 106.117.208.69_13638 -> localhost:23638"
Scrcpy 106.117.208.69_13638 -> localhost:23638


In [ ]:
"""提取 APK"""

from pathlib import Path
import re

# appName = "烈焰觉醒玩心(公测)"
# appName = "烈焰觉醒玩心(征战)"
# appName = "烈焰觉醒玩心(烈焰)"
# packageName = "com.hzqm.lyjx.z002"
# packageName = "com.wxbz.lyjxx.z009"
# packageName = "com.wxbz.lyjxx.z060"
packageName = "com.ttg.demoh5"

package_str = shell(f"pm list package -3")
package_list = list(map(lambda x: re.sub(r"package:|\s", "", x), package_str.splitlines()))
if packageName in package_list:
    packagePath = re.sub(r"package:|\s", "", shell(f"pm path {packageName}"))
    localPath = Path.home() / f"Downloads/{packageName}.apk"
    pull(packagePath, localPath)
    print(f"pull {packagePath} to {localPath}")
else:
    print(f"{packageName} 未安装")

In [ ]:
packageName = "com.tencent.mtt"
package_str = shell(f"pm list package -3")
package_list = list(map(lambda x: re.sub(r"package:|\s", "", x), package_str.splitlines()))
package_list
# shell(f"pm path {packageName}")
# re.sub(r"package:|\s", "", shell(f"pm path {packageName}"))

In [ ]:
"""批量安装"""

apk_path = "/Users/vip/Downloads/wjdr_1-11-4_107_4.apk"
DEV.install_app(apk_path)

In [ ]:
from subprocess import Popen, run, PIPE
from pathlib import Path
from multiprocessing import Process

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

cmd_ = """
adb -s {local_port} install {apk_path}
""".strip()

apk_path = "/Users/vip/Downloads/ZenlessZoneZero_2.3.apk"
cmd_list = []

local_port_list = [
    "25118",
]
for local_port in local_port_list:
    cmd = cmd_.format(local_port=f"localhost:{local_port}", apk_path=f"\"{apk_path}\"")
    cmd_list.append(cmd)

filename = Path.home() / f"Desktop/apk批量安装.sh"
filename.write_text("\n".join(cmd_list))
filename.chmod(0o700)
# Popen(['/System/Applications/Utilities/Terminal.app/Contents/MacOS/Terminal', filename])
# Popen(['open', filename])
# filename.unlink()

In [ ]:
"""开启/关闭声音"""

from subprocess import Popen, run, PIPE
from multiprocessing import Process

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')
sndcpy_cmd = """
cd ~/adb
sndcpy localhost:15647
"""
Process(target=run, args=(sndcpy_cmd.strip(),), kwargs=parameter).start()

In [ ]:
"""断开连接"""

import re
from subprocess import run, PIPE

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

exclude = [15690, 15647]
res = run("adb devices", **parameter)
devices_list_re = re.compile(r"localhost:(\d+)")
devices_list = res.stdout.strip()
port_list = devices_list_re.findall(devices_list)
port_list = filter(lambda x: int(x) not in exclude, port_list)

for port in port_list:
    cmd = f"adb disconnect localhost:{port}"
    res = run(cmd, **parameter)
    if stdout := res.stdout: print(stdout.strip())
    if stderr := res.stderr: print(stderr)

In [ ]:
"""旋转屏幕"""

orientation_func = lambda: DEV.display_info["orientation"]
orientation = orientation_func()
if orientation in (1, 3):
    DEV.shell("settings put system accelerometer_rotation 0")
    DEV.shell("settings put system user_rotation 0")
    DEV.shell("settings put system accelerometer_rotation 1")
    sleep(0.5)
    print(f"切换横屏 {orientation} --> {orientation_func()}")
else:
    DEV.shell("settings put system accelerometer_rotation 0")
    DEV.shell("settings put system user_rotation 3")
    DEV.shell("settings put system accelerometer_rotation 1")
    sleep(0.5)
    print(f"切换竖屏 {orientation} --> {orientation_func()}")

In [ ]:
"""活动"""

from airtest.core.api import *
import re
from subprocess import run, PIPE

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')

def activitiesApps(device, is_front=False):
    """
    正在活动app
    :return: applist
    """
    try:
        # 前台活动
        if is_front:
            cmd = f"adb -s {device} shell dumpsys window | grep mCurrentFocus"
            activitiesComponent = run(cmd, **parameter)
            applist = re.findall(r"mCurrentFocus.*u0\s(.*?)/", activitiesComponent.stdout)

        # 全部活动
        else:
            cmd = f"adb -s {device} shell dumpsys activity top | grep ACTIVITY"
            activitiesComponent = run(cmd, **parameter)
            applist = re.findall(r"ACTIVITY\s(.*?)/", activitiesComponent.stdout)

    except Exception as e:
        print(f"---activitiesAppsError: {e}")
        return
    return applist

device = DEV.uuid
activitiesApps(device)

In [ ]:
import re
from pprint import pprint

# cmd = f"adb -s {device} shell wm density 300"
cmd = f"adb -s {device} shell wm density 450"
run(cmd, **parameter)
# shell("pm clear com.xiaomi.gamecenter.sdk.service")
# shell(")
# shell("wm density 450")
# packageName = "com.netease.sky.bilibili"
# packageName = "com.mmbox.xbrowser"
# packageName = "com.aligames.sgzzlb.aligames"
# device_stop_app(packageName)
# device_start_app(packageName)
# applist = activityApps(0)
# pprint(applist)

In [ ]:
"""启动/关闭APP"""

DEV.start_app("com.ejoy.s3.game")

In [ ]:
from subprocess import run, PIPE

parameter = dict(stdin=PIPE, stdout=PIPE, stderr=PIPE, shell=True, encoding='utf8')


def device_start_app(device, packageName):
    cmd = f"adb -s {device} shell monkey -p \"{packageName}\" -c android.intent.category.LAUNCHER 1"
    print(cmd)
    ret = run(cmd, **parameter)
    if "No activities found to run" in ret.stdout:
        print("启动失败")

def device_stop_app(device, packageName):
    cmd = f"adb -s {device} shell am force-stop \"{packageName}\""
    run(cmd, **parameter)

device = DEV.uuid
packageName = "com.mmbox.xbrowser"
# packageName = "com.mihoyo.hyperion"
# device_start_app(device, packageName)
# device_stop_app(device, packageName)

In [ ]:
cmd = f"adb -s {device} shell am start -a android.intent.action.VIEW -d https://www.qq.com -p com.mmbox.xbrowser"
run(cmd, **parameter)

In [ ]:
"""一键释放"""

def kill_all_activity(device):
    exclude = [
        "com.teslacoilsw.launcher",
        "com.android.launcher3",
        "com.alibaba.cgs",
        "com.alibaba.android.security.activity",
    ]
    activityAppsList = list(filter(lambda x: x not in exclude, activitiesApps(device, is_front=False) or []))
    for activityApp in activityAppsList:
        device_stop_app(device, activityApp)
    if activityAppsList:
        print("已关闭以下活动", activityAppsList)

device = DEV.uuid
kill_all_activity(device)
print("当前活动", activitiesApps(device, False))

In [ ]:
"""应用列表"""

import re
import logging

logger = logging.getLogger("airtest")
logger.setLevel(logging.ERROR)

device = "localhost:20758"


def appsList():
    print(shell("pm list packages -3"))


appsList()

In [ ]:
"""下载APP"""

from subprocess import run
import pyperclip
from biz_tools import url_parse
from pathlib import Path

device = "localhost:15690"


def install_app():
    url = pyperclip.paste()
    if not url.startswith("http"):
        print("Invalid URL")
        return

    filename = Path(url_parse(url).path).name
    cmd = [
        f"adb -s {device} shell /data/curl -k -o /sdcard/Download/{filename} {url}",
        # f"adb -s {device} shell pm install /sdcard/{filename}",
        # f"adb -s {device} shell rm /sdcard/{filename}"
    ]
    for c in cmd:
        print(c)
        run(c.strip(), shell=True)


install_app()

In [ ]:
"""安装APP"""

from subprocess import run

local_port = [
    "15620",
]


def install_app():
    for port in local_port:
        cmd = f"adb -s localhost:{port} shell pm install /sdcard/base.apk && adb -s localhost:{port} shell rm /sdcard/base.apk"
        run(cmd.strip(), shell=True)
        print(f"localhost:{port} install successfully")


install_app()

In [ ]:
"""下载图片"""

import requests
from urllib import parse
from pathlib import Path
import pyperclip
import os

def download_image(url=None):
    # url = pyperclip.paste()
    url = url.split("?")[0]
    url_split = os.path.split(url)
    if not all(url_split):
        print("Invalid URL")
        return
    if not url.startswith("http"):
        print("Invalid URL")
        return

    save_path = Path("/Users/vip/Downloads") / parse.unquote(url_split[1])
    response = requests.get(url)
    save_path.write_bytes(response.content)
    # print("Save Successfully")

urls = ["https://static.jiaoyimao.com/jym-oss/resource/vmos/20250919/adc_9386221/adc_result_%E8%9C%A1%E7%83%9B_1758255266964133.jpg"]
for url in urls:
    download_image(url)

In [ ]:
"""内容拉取/推送"""

# 本地保存图片
local_filepath = "/Users/vip/Downloads/tpl20260412000307.jpg"

# 删除设备上所有图片
device_dirpath = "/sdcard/Pictures"
DEV.shell(f"rm -rf {device_dirpath} && mkdir {device_dirpath}")

# 保存我的二维码

# 推送文件
if device_filename := DEV.shell(f"ls {device_dirpath}").strip():
    remote_path = DEV.push(local_filepath, f"/sdcard/Pictures/{device_filename}")

In [ ]:
# DEV.pull("/sdcard/Android/data/com.xmcy.hykb/files/HYKB/崩坏：星穹铁道\(官服\)-4.1版本_8b9572b2268aa92054e63a396b938a31.apk", "/Users/vip/Downloads")
DEV.shell("mv /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/base.apk /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/崩坏：星穹铁道\(官服\)-4.1版本_8b9572b2268aa92054e63a396b938a31.apk")
# DEV.shell("mv /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/崩坏：星穹铁道\(官服\)-4.1版本_8b9572b2268aa92054e63a396b938a31.apk /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/base.apk")

In [ ]:
"""Airtest 滑动测试"""

from airtest.core.api import *

# auto_setup(devices=["android://localhost:5037/localhost:15888"])
# auto_setup(devices=["android://localhost:5037/localhost:15690"])
# auto_setup(devices=["android://localhost:5037/localhost:15647"])

from swipe_airtest import *

In [ ]:
start_point = (1060, 1160)

def swipe_reset():
    for _ in range(1):
        air_swipe_fast(start_point, delta_y=10000, hold_sec=0, sec=0)

swipe_reset()

In [ ]:
swipe_reset()
sp = SwipeAirtest(start_point, offset=-5)

In [ ]:
delta_y = -800
sp(delta_y=delta_y, hold_sec=0)
# sp(delta_y=-delta_y)

In [ ]:
# with SwipeAirtest(start_point, delta_y=-10000, offset=-1):
screenshot_hd()

In [ ]:
swipe_reset()
for i in range(1, 33):
    sp(delta_y=delta_y)
    print(i, delta_y)
    if i % 10 == 0:
        delta_y -= 2
    elif i % 2 == 0:
        delta_y -= 1
    else:
        delta_y = -584

In [ ]:
"""POCO 滑动测试"""

from swipe_poco import *

In [ ]:
start_point = (1340, 1140)

SwipeAirtest(start_point, 1000000)()
for i in range(10):
    SwipePoco(
        start_point,
        delta_y=-719,
        sec=2
    )()

In [ ]:
"""ADB 滑动测试"""

def adb_swipe(start_point, end_point, sec=1):
    shell(f"input swipe {start_point[0]} {start_point[1]} {end_point[0]} {end_point[1]} {sec * 1000}")

In [ ]:
"""POCO Node 测试"""

from poco.drivers.android.uiautomation import AndroidUiautomationPoco

poco = AndroidUiautomationPoco(screenshot_each_action=False)

SCREENSIZE = G.DEVICE.get_current_resolution()

from swipe_airtest import *

start_point = (650, 2000)


def swipe_reset():
    for _ in range(1):
        air_swipe_fast(start_point, delta_y=100000, hold_sec=0, sec=1)


def pos_conver(pos, mode=None):
    # 转绝对位置
    if mode == "abs":
        return int(pos[0] * SCREENSIZE[0]), int(pos[1] * SCREENSIZE[1])
    # 转相对位置
    elif mode == "rel":
        return pos[0] / SCREENSIZE[0], pos[1] / SCREENSIZE[1]
    # 转绝对位置-自适应
    elif isinstance(pos[0], float) and isinstance(pos[1], float) and \
            0 < pos[0] <= 1 and 0 < pos[1] <= 1:
        return int(pos[0] * SCREENSIZE[0]), int(pos[1] * SCREENSIZE[1])
    # 转相对位置-自适应
    else:
        return pos[0] / SCREENSIZE[0], pos[1] / SCREENSIZE[1]

In [ ]:
intro = ""
node1 = poco(nameMatches="^.+:id/tu8$").wait(3)
if node2 := node1.offspring("android.widget.TextView"):
    intro = node2.get_text().strip()

re.findall("v.*\n", intro)

In [ ]:
"""MDS-iOS 初始化"""

from swipe_ios import *

def ios_device_init():
    os.environ.update(screenRotation="v")
    ENV.iOSDeviceId = "00008101-00061DCE11C0001E"
    app = Application()
    app.start()


ios_device_init()

In [ ]:
""""MDS-iOS 滑动测试"""

def air_swipe_fast(start_point, delta_y=0.0, delta_x=0, end_point=None, offset=None, duration=None, hold_sec=0.5, sec=0.5):
    if not end_point and any((delta_x, delta_y)):
        end_point = start_point[0] + delta_x, start_point[1] + delta_y
    ios_swipe(start_point, end_point, duration=duration or 0.5, sec=sec)

In [ ]:
# 强制释放设备
force_release_device()

In [ ]:
for i in range(1, 3):
    air_swipe_fast((2254, 500), 2000, sec=0)

In [ ]:
y = -430
for i in range(1, 15):
    air_swipe_fast((2254, 900), y, sec=0, duration=1)
    print(i, y)
    # if i == 1 or i % 3 == 0:
    #     y = -256
    # else:
    #     y = -255
    # if 7 <= i <= 9:
    #     y -= 1
    sleep(2)

In [ ]:
"""MDS-iOS 高清截屏"""

save_path = Path.home().resolve() / "Downloads"
project_path = Path().resolve().parent.parent
# save_path = project_path / "iOS/ios_23016_shengchang/ios_23016_shengchang_index"

for _ in range(2):
    try:
        ios_screenshot(save_path)
        break
    except:
        ios_device_init()

In [ ]:
"""iOS云机滑动测试"""

from swipe_ios_cloud import yunyou_ios_swipe

def air_swipe_fast(start_point, delta_y=0, delta_x=0, end_point=None, offset=None, duration=None, steps=5, hold_sec=0.5, sec=0.5):
    if not end_point and any((delta_x, delta_y)):
        end_point = start_point[0] + delta_x, start_point[1] + delta_y
    yunyou_ios_swipe(start_point, end_point, duration=duration or 1.0, steps=steps or 10, sec=sec or 0.1, dev=DEV)

In [ ]:
for _ in range(15):
    air_swipe_fast((1140, 1910), delta_x=-334)

In [ ]:
"""PC 测试"""

username = "tonusnxx"
password = "e3XsS7uu"

G.DEVICE.touch((1060, 550))
sleep(0.5)
G.DEVICE.shell(f"input text \"{username}\"")
sleep(0.5)
G.DEVICE.touch((1060, 630))
sleep(0.5)
G.DEVICE.shell(f"input text \"{password}\"")
sleep(0.5)
G.DEVICE.touch((1060, 740))

In [ ]:
# 令牌
G.DEVICE.shell(f"input text 456jh")

In [ ]:
# device = "localhost:15647"
# kill_all_activity(device=device)
url = "https://cg-paas-test.aligames.com/h5demo/mobile-pc/#/cloudgame?gameId=1108&gamePlatformType=1"
browers = "com.android.chrome"
# shell(f"am start -a android.intent.action.VIEW -d \"{url}\" -p {browers}")
shell(f"am start -S -a android.intent.action.VIEW -d \"{url}\" -p {browers}")
sleep(8)
shell("input keyevent F12")

In [ ]:
shell("input keyevent F12")

In [ ]:
"""剪贴板"""

start_app("ca.zgrs.clipper")
text = shell('am broadcast -a clipper.get')
if regex_result := "".join(re.findall("data=\"(.*)\"", text)):
    print(regex_result)
keyevent("back")

In [ ]:
if text := DEV.get_clipboard():
    print(text)
    # if "InvocationTargetException" not in text:
    #     monitor_print(f"获取剪贴板文本方案1: {text}")
    #     print(text)

In [ ]:
import shlex

apk_file_dir = "/sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/"
apk_file_name = DEV.shell(f"ls {apk_file_dir}").strip()
apk_file_name = shlex.quote(apk_file_name)
DEV.shell(f"mv {apk_file_dir}{apk_file_name} {apk_file_dir}base.apk")
# DEV.shell("mv /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/base.apk /sdcard/Android/data/com.xmcy.hykb/files/HYKB/bazaar/萤火突\&击\(官服\)_dbbb3fe564cdaf40406f9694d5b9e5f9.apk")